# SynthRare — Finance CTGAN Training (Google Colab)

Trains a CTGAN model on financial time-series seed data and uploads to HuggingFace Hub.

In [ ]:
# Cell 1 — Install dependencies + mount Drive
!pip install sdv ctgan huggingface_hub pandas scikit-learn scipy
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Load seed data from Drive
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/synthrare/seed_data/finance_seed.csv')
print(f'Loaded {len(df)} rows, {len(df.columns)} columns')
df.head()

In [ ]:
# Cell 3 — Train CTGAN model
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df)

synthesizer = CTGANSynthesizer(metadata, epochs=300, verbose=True)
synthesizer.fit(df)
print('Training complete')

In [ ]:
# Cell 4 — Validate output
from scipy import stats
import numpy as np

synthetic = synthesizer.sample(num_rows=1000)
print(f'Generated {len(synthetic)} rows')

num_cols = df.select_dtypes('number').columns.tolist()
ks_scores = []
for col in num_cols:
    ks, _ = stats.ks_2samp(df[col].dropna(), synthetic[col].dropna())
    ks_scores.append(ks)
    print(f'  {col}: KS={ks:.4f}')

print(f'Mean KS statistic: {np.mean(ks_scores):.4f}  (target < 0.10)')

In [ ]:
# Cell 5 — Save and upload to HuggingFace Hub
import os
from huggingface_hub import HfApi

HF_TOKEN = 'YOUR_HF_TOKEN_HERE'  # Replace or set as Colab secret
REPO_ID = 'synthrare/finance-ctgan'

synthesizer.save('/content/model.pkl')

HfApi().upload_file(
    path_or_fileobj='/content/model.pkl',
    path_in_repo='model.pkl',
    repo_id=REPO_ID,
    token=HF_TOKEN,
)
print(f'Model uploaded to {REPO_ID}')